# Stage 4b Convolutional Autoencoder (Conv-AE)
### Alertreck · Unsupervised Anomaly Detection Paradigm

---

Conv-AE learns a compressed representation of **background audio only** (animals + wind/rain). At inference, clips that cannot be reconstructed accurately — i.e., threats, produce a high MSE reconstruction error and are flagged as anomalous.

This model answers **RQ3**: *Can unsupervised anomaly detection achieve viable threat detection performance when no labelled threat data is seen during training?*

---

## Architecture

```
Input  (1 × 128 × 301)
  │
  ├── Conv(16, 3×3) → BN → ReLU → MaxPool(2×2)     →  (16 × 64 × 150)
  ├── Conv(32, 3×3) → BN → ReLU → MaxPool(2×2)     →  (32 × 32 × 75)
  ├── Conv(64, 3×3) → BN → ReLU → MaxPool(2×2)     →  (64 × 16 × 37)
  └── Flatten → Linear(37,888 → 128)               →  latent z (128-dim)

  ├── Linear(128 → 37,888) → reshape (64 × 16 × 37)
  ├── Upsample → Conv(64→32, 3×3) → BN → ReLU      →  (32 × 32 × 75)
  ├── Upsample → Conv(32→16, 3×3) → BN → ReLU      →  (16 × 64 × 150)
  └── Upsample → Conv(16→1,  1×1) → Sigmoid        →  (1  × 128 × 301)
```

> **Note:** Bilinear upsampling with explicit target sizes avoids checkerboard artefacts and resolves the 37 → 75 asymmetry introduced by `MaxPool2d` floor-division on the 301-frame input.

---

## Training strategy

| Setting | Value |
|---|---|
| Loss | MSE (reconstruction error) |
| Optimiser | Adam · lr = 1e-3 · weight decay = 1e-4 |
| Training data | Background classes only (`background_animals`, `background_wind_rain`) |
| Curriculum | Phase A (clean + aug\_A) → Phase B (clean + aug\_B) → Phase C (clean + aug\_C) |
| Batch size | 32 |
| Early stopping | patience = 10 on val reconstruction loss |
| Augmentation | **Disabled** — reconstruction target must equal input |

---

## Anomaly detection

1. Train on background audio until convergence.
2. Score every sample in the background **validation** set with per-pixel MSE.
3. Set the anomaly threshold at the **95th percentile** of those scores.
4. At test time: MSE > threshold → anomalous (potential threat).

---

## Outputs

| File | Description |
|---|---|
| `best_model.pt` | Best checkpoint (lowest val MSE) |
| `conv_ae.onnx` | ONNX export for ONNX Runtime on Pi 4 |
| `training_curves.png` | Train / val MSE + LR schedule |
| `error_distribution.png` | Per-class reconstruction error violin plot |
| `roc_pr_curves.png` | Per-threat-class ROC and PR curves |
| `results.json` | Full metrics + training history |
| `model_config_conv_ae.json` | Config + threshold for deployment |

---

**Kaggle dataset:** `orpheusmanga/alertreck-mel2` · **Output dir:** `/kaggle/working/conv_ae`

In [4]:
import json, random, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

device: cuda


In [5]:
CFG = {
    'data_root':  Path('/kaggle/input/datasets/orpheusmanga/alertreck-processed/processed/mel'),
    'output_dir': Path('/kaggle/working/conv_ae'),
    'n_classes':  7,
    'n_mels':     128,
    'n_frames':   301,
    'batch_size': 32,
    'seed':       42,
    'label_names': [
        'background_animals', 'background_wind_rain',
        'threat_chainsaw', 'threat_dog', 'threat_gunshot',
        'threat_human', 'threat_vehicle',
    ],
    'bg_classes':          [0, 1],   # background_animals=0, background_wind_rain=1
    # Mel shards are dB-scaled, clipped to [-80, 0].
    # Normalise to [0, 1] so Sigmoid output and MSE loss are on the same scale.
    'mel_min':             -80.0,
    'mel_max':               0.0,
    'latent_dim':          256,      # increased from 128 — more capacity for background variation
    'lr':                  1e-3,
    'weight_decay':        1e-4,
    'epochs':              60,
    'patience':            10,
    'anomaly_percentile':  95,
    'phase_epochs':        {'A': 20, 'B': 15, 'C': 999},
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])

output_dir = CFG['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)
data_root = CFG['data_root']
print(f'data_root : {data_root}')
print(f'output_dir: {output_dir}')
print(f'mel range : [{CFG["mel_min"]}, {CFG["mel_max"]}] → normalised to [0, 1]')

data_root : /kaggle/input/datasets/orpheusmanga/alertreck-processed/processed/mel
output_dir: /kaggle/working/conv_ae
mel range : [-80.0, 0.0] → normalised to [0, 1]


In [6]:
MEL_MIN = CFG['mel_min']   # -80.0 dB
MEL_RNG = CFG['mel_max'] - CFG['mel_min']   # 80.0


class ShardDataset(Dataset):
    """Loads multi-sample .npz shards with X (N,128,n_frames) and y (N,) keys.

    Values are dB-scaled log-mel spectrograms clipped to [MEL_MIN, 0].
    __getitem__ normalises them to [0, 1] so Sigmoid decoder output and
    MSE loss are on the same scale — without this the training loss is ~2600
    and the model cannot converge.
    """
    def __init__(self, shard_dir: Path, augment: bool = False):
        shards = sorted(shard_dir.glob('*.npz'))
        assert shards, f'No shards found in {shard_dir}'
        xs, ys = [], []
        for s in tqdm(shards, desc=f'Loading {shard_dir.name}'):
            d = np.load(s)
            xs.append(d['X'])
            ys.append(d['y'].astype(np.int64))
        self.X = np.concatenate(xs, axis=0)
        self.y = np.concatenate(ys, axis=0)
        self.augment = augment

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        x = (x - MEL_MIN) / MEL_RNG          # [-80, 0] → [0, 1]
        x = x.clip(0.0, 1.0)                 # guard against any out-of-range values
        return torch.from_numpy(x).unsqueeze(0).float(), \
               torch.tensor(self.y[idx], dtype=torch.long)


def bg_indices(dataset):
    """Return flat indices of background-class samples (handles ShardDataset and ConcatDataset)."""
    bg = set(CFG['bg_classes'])
    if hasattr(dataset, 'y'):
        return [i for i, lbl in enumerate(dataset.y) if int(lbl) in bg]
    idx, offset = [], 0
    for ds in dataset.datasets:
        for i, lbl in enumerate(ds.y):
            if int(lbl) in bg:
                idx.append(offset + i)
        offset += len(ds)
    return idx


# Load all splits — augment=False: clean input = clean reconstruction target
val_ds  = ShardDataset(data_root / 'val',  augment=False)
test_ds = ShardDataset(data_root / 'test', augment=False)
train_clean_ds = ShardDataset(data_root / 'train',       augment=False)
train_aug_A_ds = ShardDataset(data_root / 'train_aug_A', augment=False)
train_aug_B_ds = ShardDataset(data_root / 'train_aug_B', augment=False)
train_aug_C_ds = ShardDataset(data_root / 'train_aug_C', augment=False)

phase_full = {
    'A': ConcatDataset([train_clean_ds, train_aug_A_ds]),
    'B': ConcatDataset([train_clean_ds, train_aug_B_ds]),
    'C': ConcatDataset([train_clean_ds, train_aug_C_ds]),
}

phase_bg = {ph: Subset(ds, bg_indices(ds)) for ph, ds in phase_full.items()}
val_bg   = Subset(val_ds, bg_indices(val_ds))

# Sanity-check: confirm normalised range
_x_sample, _ = val_ds[0]
print(f'Normalised sample range: min={_x_sample.min():.4f}  max={_x_sample.max():.4f}  (expected ≈ 0–1)')
print(f'Background train Phase A: {len(phase_bg["A"]):,}')
print(f'Background train Phase B: {len(phase_bg["B"]):,}')
print(f'Background train Phase C: {len(phase_bg["C"]):,}')
print(f'Background val          : {len(val_bg):,}')
print(f'Test (all classes)      : {len(test_ds):,}')

Loading train_aug_C: 100%|██████████| 45/45 [00:37<00:00,  1.19it/s]


Normalised sample range: min=0.0000  max=1.0000  (expected ≈ 0–1)
Background train Phase A: 14,496
Background train Phase B: 21,744
Background train Phase C: 28,992
Background val          : 3,221
Test (all classes)      : 5,925


In [7]:
class ConvAE(nn.Module):
    """
    Convolutional autoencoder for background audio.

    Input is normalised to [0, 1] (from dB-scaled log-mel [-80, 0]).
    Sigmoid decoder output is therefore on the same scale as the input,
    and MSE loss values will be in [0, 1] rather than the thousands.

    Encoder:  (1,128,301) → 3× [Conv→BN→ReLU→MaxPool2d] → (64,16,37) → Dropout → FC → 256-dim latent
    Decoder:  256-dim → FC → (64,16,37) → 3× [Upsample→Conv→BN→ReLU] → Conv(1×1) → Sigmoid → (1,128,301)
    """
    _ENC_C, _ENC_H, _ENC_W = 64, 16, 37

    def __init__(self, latent_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        C, H, W = self._ENC_C, self._ENC_H, self._ENC_W

        self.enc_cnn = nn.Sequential(
            nn.Conv2d(1,  16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.fc_enc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(C * H * W, latent_dim),
        )
        self.fc_dec = nn.Linear(latent_dim, C * H * W)
        self.dec_cnn = nn.Sequential(
            nn.Upsample(size=(32, 75),   mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Upsample(size=(64, 150),  mode='bilinear', align_corners=False),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Upsample(size=(128, 301), mode='bilinear', align_corners=False),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc_cnn(x).flatten(1)
        return self.fc_enc(h)

    def decode(self, z):
        C, H, W = self._ENC_C, self._ENC_H, self._ENC_W
        h = self.fc_dec(z).view(-1, C, H, W)
        return self.dec_cnn(h)

    def forward(self, x):
        return self.decode(self.encode(x))


model = ConvAE(latent_dim=CFG['latent_dim']).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'ConvAE — {n_params:,} parameters  |  latent_dim={CFG["latent_dim"]}')

with torch.no_grad():
    dummy = torch.rand(2, 1, CFG['n_mels'], CFG['n_frames'], device=device)  # [0,1] range
    out   = model(dummy)
    assert out.shape == dummy.shape, f'Shape mismatch: {out.shape} vs {dummy.shape}'
    assert out.min() >= 0.0 and out.max() <= 1.0, 'Decoder output out of [0,1]'
print(f'Shape check passed: {list(out.shape)}  |  output range [{out.min():.3f}, {out.max():.3f}]')

ConvAE — 19,483,521 parameters  |  latent_dim=256
Shape check passed: [2, 1, 128, 301]  |  output range [0.250, 0.763]


In [8]:
# ── Hyperparameter search (Optuna) — select on VALIDATION DETECTION AUC ────────
# Conv-AE trains on background only. The val split's threat labels are used purely
# to SCORE candidate configs (model selection), never for training. A short proxy
# (PROXY_EPOCHS on Phase-A background) + median pruning keeps the search cheap.
try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna

USE_OPTUNA   = True
N_TRIALS     = 15
PROXY_EPOCHS = 10

# all-class val loader (background + threats) — the detection-AUC selection metric
val_all_loader = DataLoader(val_ds, batch_size=64, shuffle=False,
                            num_workers=2, pin_memory=True)
val_is_threat  = (~np.isin(val_ds.y, list(CFG['bg_classes']))).astype(int)


@torch.no_grad()
def val_detection_auc(m) -> float:
    """Background-vs-threat ROC-AUC of per-sample reconstruction MSE on the val split."""
    m.eval()
    errs = []
    for x, _ in val_all_loader:
        x = x.to(device)
        errs.append(((m(x) - x) ** 2).mean(dim=[1, 2, 3]).cpu().numpy())
    return float(roc_auc_score(val_is_threat, np.concatenate(errs)))


def _objective(trial):
    latent = trial.suggest_categorical('latent_dim', [128, 256, 384, 512])
    lr     = trial.suggest_float('lr', 3e-4, 3e-3, log=True)
    drop   = trial.suggest_float('dropout', 0.1, 0.4)
    wd     = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    m   = ConvAE(latent_dim=latent, dropout=drop).to(device)
    opt = optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(phase_bg['A'], batch_size=CFG['batch_size'], shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    best = 0.0
    for ep in range(1, PROXY_EPOCHS + 1):
        m.train()
        for x, _ in loader:
            x = x.to(device)
            loss = F.mse_loss(m(x), x)
            opt.zero_grad(); loss.backward(); opt.step()
        auc = val_detection_auc(m)
        best = max(best, auc)
        trial.report(auc, ep)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best


if USE_OPTUNA:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=CFG['seed']),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    )
    t0 = time.time()
    study.optimize(_objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f'Optuna done in {time.time()-t0:.0f}s | {len(study.trials)} trials '
          f'({sum(t.state.name == "PRUNED" for t in study.trials)} pruned)')
    print(f'Best val detection AUC = {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')

    bp = study.best_params
    CFG['latent_dim']   = bp['latent_dim']
    CFG['lr']           = bp['lr']
    CFG['dropout']      = bp['dropout']
    CFG['weight_decay'] = bp['weight_decay']

    (output_dir / 'optuna_conv_ae.json').write_text(json.dumps({
        'selection_metric': 'val_detection_auc',
        'n_trials': N_TRIALS, 'proxy_epochs': PROXY_EPOCHS,
        'best_value': float(study.best_value), 'best_params': bp,
        'trials': [{'number': t.number, 'value': t.value,
                    'params': t.params, 'state': t.state.name} for t in study.trials],
    }, indent=2))
    print(f"Search saved -> {output_dir / 'optuna_conv_ae.json'}")
else:
    CFG.setdefault('dropout', 0.2)

# rebuild the model with the tuned hyperparameters (the training cell uses `model`)
model = ConvAE(latent_dim=CFG['latent_dim'], dropout=CFG['dropout']).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nTuned ConvAE — {n_params:,} params | latent_dim={CFG["latent_dim"]} '
      f"dropout={CFG['dropout']:.2f} lr={CFG['lr']:.2e} wd={CFG['weight_decay']:.1e}")

  0%|          | 0/15 [00:00<?, ?it/s]

Optuna done in 4284s | 15 trials (0 pruned)
Best val detection AUC = 0.8126
Best params: {'latent_dim': 384, 'lr': 0.0017569008478550484, 'dropout': 0.2203452469088676, 'weight_decay': 0.0008244137703704573}
Search saved -> /kaggle/working/conv_ae/optuna_conv_ae.json

Tuned ConvAE — 29,182,977 params | latent_dim=384 dropout=0.22 lr=1.76e-03 wd=8.2e-04


In [9]:
optimizer = optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)

phase_end = {
    'A': CFG['phase_epochs']['A'],
    'B': CFG['phase_epochs']['A'] + CFG['phase_epochs']['B'],
}

def get_phase(epoch: int) -> str:
    if epoch <= phase_end['A']: return 'A'
    if epoch <= phase_end['B']: return 'B'
    return 'C'


def run_epoch(loader, train: bool = True) -> float:
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, _ in loader:
            x = x.to(device)
            recon = model(x)
            loss  = F.mse_loss(recon, x)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * x.size(0)
            n     += x.size(0)
    return total / n


val_loader = DataLoader(val_bg, batch_size=CFG['batch_size'],
                        shuffle=False, num_workers=2, pin_memory=True)

# Best-model selection is on VAL DETECTION AUC (consistent with the Optuna search),
# not val reconstruction loss — the lowest-loss AE is not the best anomaly detector.
history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'lr': [], 'phase': []}
best_val_auc, best_val_loss, best_epoch, no_improve = -1.0, float('inf'), 0, 0

print(f'Starting training — {CFG["epochs"]} epochs | select on val detection AUC | device: {device}')
t0 = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    phase        = get_phase(epoch)
    train_loader = DataLoader(phase_bg[phase], batch_size=CFG['batch_size'],
                              shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    tr_loss = run_epoch(train_loader, train=True)
    va_loss = run_epoch(val_loader,   train=False)
    va_auc  = val_detection_auc(model)          # background-vs-threat AUC on val
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_auc'].append(va_auc)
    history['lr'].append(scheduler.get_last_lr()[0])
    history['phase'].append(phase)

    marker = ''
    if va_auc > best_val_auc:
        best_val_auc, best_val_loss, best_epoch = va_auc, va_loss, epoch
        torch.save({'epoch': epoch, 'phase': phase,
                    'model_state': model.state_dict(),
                    'optim_state': optimizer.state_dict(),
                    'val_loss': va_loss, 'val_auc': va_auc, 'cfg': CFG},
                   output_dir / 'best_model.pt')
        marker = '  >> Saved best (AUC)'
        no_improve = 0
    else:
        no_improve += 1

    elapsed = time.time() - t0
    print(f'Ep {epoch:03d}/{CFG["epochs"]} [{phase}] '
          f'tr_loss={tr_loss:.6f} va_loss={va_loss:.6f} va_auc={va_auc:.4f} '
          f'lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.1f}s){marker}')

    if no_improve >= CFG['patience']:
        print(f'Early stop at epoch {epoch} (no val-AUC gain for {CFG["patience"]} epochs)')
        break

    if epoch == phase_end['A']:
        print(f'  >> Switched to Phase B at epoch {epoch + 1}')
    elif epoch == phase_end['B']:
        print(f'  >> Switched to Phase C at epoch {epoch + 1}')

print(f'\nBest epoch: {best_epoch}  |  best val AUC: {best_val_auc:.4f}  '
      f'(val_loss at best = {best_val_loss:.6f})')

Starting training — 60 epochs | select on val detection AUC | device: cuda
Ep 001/60 [A] tr_loss=0.016053 va_loss=0.018487 va_auc=0.7907 lr=1.76e-03 (31.6s)  >> Saved best (AUC)
Ep 002/60 [A] tr_loss=0.013823 va_loss=0.017822 va_auc=0.7733 lr=1.75e-03 (62.5s)
Ep 003/60 [A] tr_loss=0.011990 va_loss=0.013532 va_auc=0.7785 lr=1.75e-03 (93.3s)
Ep 004/60 [A] tr_loss=0.011193 va_loss=0.011778 va_auc=0.8032 lr=1.74e-03 (124.8s)  >> Saved best (AUC)
Ep 005/60 [A] tr_loss=0.010813 va_loss=0.017597 va_auc=0.7603 lr=1.73e-03 (155.3s)
Ep 006/60 [A] tr_loss=0.011380 va_loss=0.008587 va_auc=0.7938 lr=1.71e-03 (185.8s)
Ep 007/60 [A] tr_loss=0.011004 va_loss=0.016978 va_auc=0.7771 lr=1.70e-03 (216.1s)
Ep 008/60 [A] tr_loss=0.011708 va_loss=0.011415 va_auc=0.8014 lr=1.68e-03 (246.4s)
Ep 009/60 [A] tr_loss=0.011072 va_loss=0.014280 va_auc=0.7457 lr=1.66e-03 (276.7s)
Ep 010/60 [A] tr_loss=0.010390 va_loss=0.009523 va_auc=0.7994 lr=1.64e-03 (306.8s)
Ep 011/60 [A] tr_loss=0.010234 va_loss=0.013739 va_auc=0

In [10]:
def shade_phases(ax, phases):
    colors = {'A': '#e8f5e9', 'B': '#fff3e0', 'C': '#fce4ec'}
    seen, prev_ph, start = set(), phases[0], 0
    for i, ph in enumerate(phases):
        if ph != prev_ph:
            label = f'Phase {prev_ph}' if prev_ph not in seen else None
            ax.axvspan(start + 1, i, alpha=0.35, color=colors[prev_ph], label=label)
            seen.add(prev_ph)
            start, prev_ph = i, ph
    label = f'Phase {prev_ph}' if prev_ph not in seen else None
    ax.axvspan(start + 1, len(phases), alpha=0.35, color=colors[prev_ph], label=label)


epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ax = axes[0]
ax.plot(epochs_range, history['train_loss'], label='Train MSE')
ax.plot(epochs_range, history['val_loss'],   label='Val MSE')
shade_phases(ax, history['phase'])
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best ep {best_epoch}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss'); ax.set_title('Reconstruction Loss')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs_range, history['val_auc'], color='#6A1B9A', label='Val detection AUC')
shade_phases(ax, history['phase'])
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.7,
           label=f'Best ep {best_epoch} (AUC={best_val_auc:.3f})')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.6, label='Chance (0.5)')
ax.set_xlabel('Epoch'); ax.set_ylabel('ROC-AUC (background vs threat)')
ax.set_title('Validation Detection AUC  (selection metric)')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(epochs_range, history['lr'])
shade_phases(ax, history['phase'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning rate (log)'); ax.set_title('LR Schedule')
ax.set_yscale('log'); ax.grid(True, alpha=0.3); ax.legend()

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

Saved training_curves.png


In [11]:
# Load best checkpoint
# weights_only=False is safe here — checkpoint was written by this notebook.
# weights_only=True fails on PyTorch 2.6+ because cfg stores pathlib.Path objects.
import pathlib
torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.Path])
ckpt = torch.load(output_dir / 'best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint: epoch {ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.6f}')


def mse_scores(dataset, desc='Scoring') -> tuple[np.ndarray, np.ndarray]:
    """Return (per-sample MSE scores, labels) for an entire dataset."""
    loader = DataLoader(dataset, batch_size=CFG['batch_size'],
                        shuffle=False, num_workers=2, pin_memory=True)
    scores, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc):
            x     = x.to(device)
            recon = model(x)
            mse   = F.mse_loss(recon, x, reduction='none').mean(dim=[1, 2, 3])
            scores.append(mse.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(scores), np.concatenate(labels)


# Fit threshold on background validation set
bg_val_scores, _ = mse_scores(val_bg, 'Threshold (val bg)')
threshold = np.percentile(bg_val_scores, CFG['anomaly_percentile'])

print(f'\nBackground val MSE  mean={bg_val_scores.mean():.6f}  std={bg_val_scores.std():.6f}')
print(f'Anomaly threshold (p{CFG["anomaly_percentile"]}): {threshold:.6f}')

Loaded checkpoint: epoch 4  val_loss=0.011778


Threshold (val bg): 100%|██████████| 101/101 [00:01<00:00, 56.50it/s]


Background val MSE  mean=0.011778  std=0.005286
Anomaly threshold (p95): 0.022087


In [12]:
test_scores, test_labels = mse_scores(test_ds, 'Test scoring')

label_names  = CFG['label_names']
bg_mask      = test_labels < 2           # True for background test samples
threat_mask  = ~bg_mask                  # True for threat test samples

# ── Binary anomaly detection (all threats = 1, all backgrounds = 0) ──────────
binary_labels = threat_mask.astype(int)
binary_auc    = roc_auc_score(binary_labels, test_scores)
binary_ap     = average_precision_score(binary_labels, test_scores)
preds         = (test_scores > threshold).astype(int)
binary_acc    = (preds == binary_labels).mean()
tp  = ((preds == 1) & (binary_labels == 1)).sum()
fp  = ((preds == 1) & (binary_labels == 0)).sum()
fn  = ((preds == 0) & (binary_labels == 1)).sum()
tn  = ((preds == 0) & (binary_labels == 0)).sum()
tpr = tp / (tp + fn + 1e-8)
fpr_rate = fp / (fp + tn + 1e-8)

print('Binary anomaly detection (threats vs backgrounds):')
print(f'  AUC-ROC  : {binary_auc:.4f}')
print(f'  Avg Prec : {binary_ap:.4f}')
print(f'  Accuracy : {binary_acc:.4f}  (threshold=p{CFG["anomaly_percentile"]})')
print(f'  TPR      : {tpr:.4f}')
print(f'  FPR      : {fpr_rate:.4f}')

# ── Per-class AUC-ROC (each threat class vs all background test samples) ──────
per_class_auc, per_class_ap = {}, {}
for c in range(2, CFG['n_classes']):
    mask  = bg_mask | (test_labels == c)
    y_bin = (test_labels[mask] == c).astype(int)
    s     = test_scores[mask]
    try:
        per_class_auc[label_names[c]] = roc_auc_score(y_bin, s)
        per_class_ap[label_names[c]]  = average_precision_score(y_bin, s)
    except Exception:
        per_class_auc[label_names[c]] = float('nan')
        per_class_ap[label_names[c]]  = float('nan')

print('\nPer-class AUC-ROC (threat vs background):')
for name in per_class_auc:
    print(f'  {name:<30} AUC={per_class_auc[name]:.4f}  AP={per_class_ap[name]:.4f}')

Test scoring: 100%|██████████| 186/186 [00:03<00:00, 59.43it/s]

Binary anomaly detection (threats vs backgrounds):
  AUC-ROC  : 0.8050
  Avg Prec : 0.7580
  Accuracy : 0.6717  (threshold=p95)
  TPR      : 0.3433
  FPR      : 0.0539

Per-class AUC-ROC (threat vs background):
  threat_chainsaw                AUC=0.7577  AP=0.2482
  threat_dog                     AUC=0.6967  AP=0.1456
  threat_gunshot                 AUC=0.8386  AP=0.5320
  threat_human                   AUC=0.9567  AP=0.7803
  threat_vehicle                 AUC=0.4035  AP=0.0536


In [13]:
# Reconstruction error distribution — violin plot per class
fig, ax = plt.subplots(figsize=(13, 5))

data_by_class = [test_scores[test_labels == c] for c in range(CFG['n_classes'])]
colors = ['#4CAF50' if c < 2 else '#F44336' for c in range(CFG['n_classes'])]

parts = ax.violinplot(data_by_class, positions=range(CFG['n_classes']),
                      showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i]); pc.set_alpha(0.55)

ax.axhline(threshold, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold (p{CFG["anomaly_percentile"]} = {threshold:.4f})')
ax.set_xticks(range(CFG['n_classes']))
ax.set_xticklabels([n.replace('_', '\n') for n in label_names], fontsize=8)
ax.set_ylabel('MSE Reconstruction Error')
ax.set_title('Reconstruction Error Distribution by Class\n(green = background, red = threat)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved error_distribution.png')

Saved error_distribution.png


In [14]:
# ROC and PR curves — per threat class vs background
threat_names = [n for n in label_names if n.startswith('threat')]
colors_t     = plt.cm.Set2(np.linspace(0, 1, len(threat_names)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for name, col in zip(threat_names, colors_t):
    c    = label_names.index(name)
    mask = bg_mask | (test_labels == c)
    y_b  = (test_labels[mask] == c).astype(int)
    fpr_v, tpr_v, _ = roc_curve(y_b, test_scores[mask])
    ax.plot(fpr_v, tpr_v, color=col, label=f'{name} ({per_class_auc[name]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curves (per threat class vs background)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = axes[1]
for name, col in zip(threat_names, colors_t):
    c    = label_names.index(name)
    mask = bg_mask | (test_labels == c)
    y_b  = (test_labels[mask] == c).astype(int)
    prec_v, rec_v, _ = precision_recall_curve(y_b, test_scores[mask])
    ax.plot(rec_v, prec_v, color=col, label=f'{name} ({per_class_ap[name]:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR Curves (per threat class vs background)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved roc_pr_curves.png')

Saved roc_pr_curves.png


In [15]:
model.eval()
dummy     = torch.zeros(1, 1, CFG['n_mels'], CFG['n_frames'], device=device)
onnx_path = str(output_dir / 'conv_ae.onnx')

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['mel'],
    output_names=['reconstruction'],
    dynamic_axes={'mel': {0: 'batch'}, 'reconstruction': {0: 'batch'}},
    opset_version=17,
    dynamo=False,       # force legacy exporter — dynamo path requires onnxscript
)
print(f'ONNX exported → {onnx_path}')

/tmp/ipykernel_58/3505089723.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX exported → /kaggle/working/conv_ae/conv_ae.onnx


In [16]:
results = {
    'model':              'ConvAE',
    'best_epoch':         best_epoch,
    'best_val_auc':       float(best_val_auc),
    'best_val_loss':      best_val_loss,
    'selection_metric':   'val_detection_auc',
    'optuna_best_params': (study.best_params if 'study' in globals() else None),
    'latent_dim':         CFG['latent_dim'],
    'lr':                 CFG['lr'],
    'weight_decay':       CFG['weight_decay'],
    'dropout':            CFG.get('dropout'),
    'n_params':           n_params,
    'anomaly_threshold':  float(threshold),
    'anomaly_percentile': CFG['anomaly_percentile'],
    'binary_auc':         float(binary_auc),
    'binary_ap':          float(binary_ap),
    'binary_acc':         float(binary_acc),
    'tpr_at_threshold':   float(tpr),
    'fpr_at_threshold':   float(fpr_rate),
    'per_class_auc':      {k: float(v) for k, v in per_class_auc.items()},
    'per_class_ap':       {k: float(v) for k, v in per_class_ap.items()},
    'history':            history,
}

with open(output_dir / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

model_cfg = {
    'n_classes':          CFG['n_classes'],
    'n_mels':             CFG['n_mels'],
    'n_frames':           CFG['n_frames'],
    'latent_dim':         CFG['latent_dim'],
    'label_names':        CFG['label_names'],
    'anomaly_threshold':  float(threshold),
    'anomaly_percentile': CFG['anomaly_percentile'],
    'best_epoch':         best_epoch,
    'best_val_auc':       float(best_val_auc),
    'best_val_loss':      best_val_loss,
    'binary_auc':         float(binary_auc),
}

with open(output_dir / 'model_config_conv_ae.json', 'w') as f:
    json.dump(model_cfg, f, indent=2)

print('Saved results.json and model_config_conv_ae.json')
print(f'\n{"─"*45}')
print(f'  Best val detection AUC : {best_val_auc:.4f}  (epoch {best_epoch})')
print(f'  Binary AUC-ROC (test)  : {binary_auc:.4f}')
print(f'  Binary Avg Precision   : {binary_ap:.4f}')
print(f'  Accuracy @ p{CFG["anomaly_percentile"]}          : {binary_acc:.4f}')
print(f'  TPR (sensitivity)      : {tpr:.4f}')
print(f'  FPR                    : {fpr_rate:.4f}')
print(f'{"─"*45}')

Saved results.json and model_config_conv_ae.json

─────────────────────────────────────────────
  Best val detection AUC : 0.8032  (epoch 4)
  Binary AUC-ROC (test)  : 0.8050
  Binary Avg Precision   : 0.7580
  Accuracy @ p95          : 0.6717
  TPR (sensitivity)      : 0.3433
  FPR                    : 0.0539
─────────────────────────────────────────────
